In [1]:
import hydra
from omegaconf import OmegaConf
import numpy as np
import torch
from matplotlib import pyplot as plt
from segmentation.environment import SegmentationEnv
from segmentation.learners import DQNLearner

/home/stefanob00/miniconda3/envs/real4cpd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
hydra.initialize(config_path="conf", version_base=None)
cfg = hydra.compose(config_name="config")

print(OmegaConf.to_yaml(cfg))

root_folder: data/honeybee/
dataset_name: honeybee
dataset: data/honeybee/sequence3.npz
output: dqn_final
checkpoint_root: checkpoints/ckp_honeybee_final.pt
opt_max_calls: 25
budget: 30
tolerance: 15
level_wavelet: 3
window_size: 30
gamma: 0.9
tau: 0.001
initial_exploration: 0.7
exploration_decay: 0.9982
minimum_exploration: 0.01
dropout: 0.05
lr: 0.0005
weight_decay: 0.01
betas:
- 0.9
- 0.999
batch_size: 32
buffer_capacity: 10000
pool_size: 16
num_episodes: 4000
num_cpus: 6



In [3]:
metric = {"precision": np.zeros(31), "recall": np.zeros(31), "f1": np.zeros(31)}
for n_data in range(5):
    for _ in range(5):
        data = np.load(f"data/honeybee/sequence{n_data+1}.npz")
        samples = data["samples"]
        gt_break_points = data["gt_break_points"]
        
        env = SegmentationEnv(
                samples=samples, 
                gt_break_points=gt_break_points,
                level_wavelet=cfg.level_wavelet,
                window_size=cfg.window_size,
                tolerance=cfg.tolerance,
                args=cfg) 

        state, info = env.reset()
        learner = DQNLearner(num_samples=samples.shape[0], gt_break_points=gt_break_points, args=cfg, options={"mode": 'eval'})
        metric["precision"][0] += info["precision"]
        metric["recall"][0] += info["recall"]
        metric["f1"][0] += info["f1"]

        learner.model.load_state_dict(torch.load(cfg.checkpoint_root))

        break_points = env.segmenter.get_break_points()
        
        total_reward = 0
        done = False
        step = 0
        while step < 30:
            action = learner.get_candidate(state)
            if not done:
                next_state, reward, done, terminated, info = env.step(action)
            total_reward += reward
            state = next_state
            print(f"Step Reward {step}: {reward:.4f}, Precision: {info['precision']:.4f}, Recall: {info['recall']:.4f}, F1: {info['f1']:.4f}")
            step += 1
            metric["precision"][step] += info["precision"]
            metric["recall"][step] += info["recall"]
            metric["f1"][step] += info["f1"]


# Average over data
for key in metric.keys():
    metric[key] /= 25

# save metrics
np.savez(f"results/honeybee.npz", precision=metric["precision"], recall=metric["recall"], f1=metric["f1"])

Step Reward 0: -0.0612, Precision: 0.4783, Recall: 0.5000, F1: 0.4889
Step Reward 1: -0.1719, Precision: 0.7500, Recall: 0.2727, F1: 0.4000
Step Reward 2: 0.1267, Precision: 0.8750, Recall: 0.3182, F1: 0.4667
Step Reward 3: -0.1485, Precision: 0.6667, Recall: 0.2727, F1: 0.3871
Step Reward 4: 0.0924, Precision: 0.7000, Recall: 0.3182, F1: 0.4375
Step Reward 5: 0.0000, Precision: 0.7000, Recall: 0.3182, F1: 0.4375
Step Reward 6: -0.0890, Precision: 0.6667, Recall: 0.2727, F1: 0.3871
Step Reward 7: -0.0932, Precision: 0.6250, Recall: 0.2273, F1: 0.3333
Step Reward 8: 0.0914, Precision: 0.6667, Recall: 0.2727, F1: 0.3871
Step Reward 9: -0.0202, Precision: 0.6000, Recall: 0.2727, F1: 0.3750
Step Reward 10: 0.0804, Precision: 0.6364, Recall: 0.3182, F1: 0.4242
Step Reward 11: 0.0742, Precision: 0.6667, Recall: 0.3636, F1: 0.4706
Step Reward 12: 0.0685, Precision: 0.6923, Recall: 0.4091, F1: 0.5143
Step Reward 13: 0.0876, Precision: 0.7692, Recall: 0.4545, F1: 0.5714
Step Reward 14: 0.0595, 